# Descart Benchmark for spatial mutliomcis data integration on simulated dataset

Notebook benchmarks spatial mutliomcis data integration using Descart on simulated dataset.

## Loading

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from Descart_utlis import *
import omicverse as ov
import anndata as ad
import pandas as pd
import scanpy as sc
import numpy as np

## Descart pipeline

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import scale
import scipy
import time

# Assuming run_descart is a function defined elsewhere
# from your_module import run_descart

# Set the directory for the datasets and the output directory
data_dir = 'Original_Simulated_Data'
output_dir = 'Processed_Simulated_Data'
os.makedirs(output_dir, exist_ok=True)

# Loop through each dataset
for i in range(1, 6):
    print(f"Process {data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad data.")
    # Read the RNA and ATAC datasets
    adata_rna = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad')
    adata_atac = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_atac.h5ad')
    sc.pp.filter_genes(adata_atac, min_cells=1)
    adata_atac.obsm['spatial'] = adata_rna.obsm['spatial']

    # Parameters for DESCART
    num_select_peak = 10000
    seed_base = 1
    tf = 'tfidf2'
    pc = 10
    k = 20
    similarity = 'cosine'
    iter_time = 4
    spmethod = 'threshold'
    neighbor = 5
    sp_dist = 'recip'
    pre_select = 'highest'
    peaks_num = 50000
    distance = 'euclidean'
    r = 0.4

    start_time = time.time()

    # Run DESCART
    adata = sc.AnnData(adata_atac.X, dtype='float32')
    idx, sorted_index, simi_matrix, idx_all, scores, \
    selected_peaks_data, similarity_matrix_acb, similarity_matrix_spatial = run_descart(adata_atac, 
                                                                                       num_select_peak, 
                                                                                       seed_base=seed_base, 
                                                                                       tfidf=tf, 
                                                                                       ifPCA=True, 
                                                                                       pc=pc, k=k, 
                                                                                       similarity=similarity, 
                                                                                       iters=iter_time, 
                                                                                       spmethod=spmethod,
                                                                                       neighbor=neighbor,
                                                                                       sp_dist=sp_dist, 
                                                                                       pre_select=pre_select,
                                                                                       peaks_num=peaks_num, 
                                                                                       distance=distance, 
                                                                                       r=r)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Running time for dataset {i}: {elapsed_time:.6f} seconds")

    # Enhance data and perform PCA
    data_enhance = scale(simi_matrix @ selected_peaks_data)
    count_filter = PCA(n_components=10, random_state=int(seed_base*1000)).fit_transform(np.array(data_enhance))
    adata_pca = sc.AnnData(scipy.sparse.csc_matrix(count_filter), dtype='float32')
    adata_pca.obs['cell_type'] = list(adata_rna.obs['cell_type'])
    adata_pca.obsm['x_pca'] = adata_pca.X.toarray()
    adata_pca.obsm['spatial'] = adata_atac.obsm['spatial']

    # Compute neighbors and perform clustering
    ov.pp.neighbors(adata_pca, n_neighbors=15, n_pcs=10, use_rep='x_pca')
    ov.utils.cluster(adata_pca, use_rep='x_pca', method='leiden', resolution=0.15)

    # Plot spatial clustering results
    sc.pl.spatial(adata_pca, color=['cell_type', 'leiden'], spot_size=0.12, wspace=0.4)

    # Save the processed dataset
    output_path = f'{output_dir}/Simulated_Dataset_{i}/Descart_atac.h5ad'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    adata_pca.write_h5ad(output_path, compression='gzip')

In [ ]:
!pip list